In [4]:
import argparse
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
import re

def basic_tokens(text: str):
    if not isinstance(text, str): text = ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []

def load_corpus(path, text_cols):
    df = pd.read_csv(path, encoding="utf-8")
    # ensure id is str
    if "id" in df.columns:
        df["id"] = df["id"].astype(str)
    # build doc
    parts = [df[c].fillna("").astype(str) if c in df.columns else pd.Series([""]*len(df)) for c in text_cols]
    doc = parts[0]
    for p in parts[1:]:
        doc = doc + " " + p
    df["doc"] = doc
    tokens = df["doc"].apply(basic_tokens).tolist()
    return df, tokens

def average_precision_at_k(ranked_ids, relevant_set, k):
    if not relevant_set:
        return 0.0
    hits = 0
    sum_prec = 0.0
    for i, doc in enumerate(ranked_ids[:k], start=1):
        if doc in relevant_set:
            hits += 1
            sum_prec += hits / i
    return sum_prec / len(relevant_set)

def dcg_at_k(rels, k):
    return sum((2**r - 1) / math.log2(i+1+1) for i, r in enumerate(rels[:k]))

def ndcg_at_k(ranked_ids, rels_map, k):
    rels = [rels_map.get(d, 0) for d in ranked_ids[:k]]
    dcg = dcg_at_k(rels, k)
    ideal = sorted(rels_map.values(), reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_bm25(df, corpus_tokens, qrels_df, k1, b, topk=10):
    bm25 = BM25Okapi(corpus_tokens, k1=k1, b=b)
    queries = qrels_df["query"].unique()
    ap_list = []
    ndcg_list = []
    for q in queries:
        q_tokens = basic_tokens(q)
        if not q_tokens:
            continue
        scores = bm25.get_scores(q_tokens)
        top_idx = np.argsort(scores)[::-1][:topk]
        ranked_ids = df.iloc[top_idx]["id"].astype(str).tolist()
        sub = qrels_df[qrels_df["query"] == q]
        rel_map = dict(zip(sub["doc_id"].astype(str), sub["relevance"].astype(int)))
        relevant_set = {d for d, r in rel_map.items() if r > 0}
        ap = average_precision_at_k(ranked_ids, relevant_set, topk)
        ndcg = ndcg_at_k(ranked_ids, rel_map, topk)
        ap_list.append(ap)
        ndcg_list.append(ndcg)
    return np.mean(ap_list) if ap_list else 0.0, np.mean(ndcg_list) if ndcg_list else 0.0

def grid_search(df, corpus_tokens, qrels_df, k1_range, b_range, topk=10):
    rows = []
    for k1, b in itertools.product(k1_range, b_range):
        map_k, ndcg_k = evaluate_bm25(df, corpus_tokens, qrels_df, k1, b, topk=topk)
        rows.append({"k1": k1, "b": b, "MAP@k": map_k, "NDCG@k": ndcg_k})
        print(f"k1={k1:.3f} b={b:.3f} => MAP@{topk}={map_k:.4f} NDCG@{topk}={ndcg_k:.4f}")
    res = pd.DataFrame(rows)
    return res
if __name__ == "__main__":
    import os
    p = argparse.ArgumentParser()
    p.add_argument("--corpus", default="../output/aksesoriAnak_enriched.csv",
                   help="path to corpus csv")
    p.add_argument("--qrels", default="../output/qrels.csv",
                   help="path to qrels csv (columns: query,doc_id,relevance)")
    p.add_argument("--topk", type=int, default=10)
    # ketika dijalankan di notebook, ipykernel mengirim argumen -> tangani SystemExit
    try:
        args = p.parse_args()
    except SystemExit:
        args = p.parse_args([])

    if not os.path.exists(args.corpus):
        raise FileNotFoundError(f"Corpus not found: {args.corpus}")
    if not os.path.exists(args.qrels):
        raise FileNotFoundError(f"Qrels not found: {args.qrels} — buat file qrels.csv atau berikan --qrels")

    # customize fields if needed
    TEXT_COLS = ["name", "category_breadcrumb", "shop_city"]

    df, corpus_tokens = load_corpus(args.corpus, TEXT_COLS)
    qrels = pd.read_csv(args.qrels, encoding="utf-8")

    # coarse ranges
    k1_coarse = np.linspace(0.5, 2.0, 7)
    b_coarse = np.linspace(0.0, 1.0, 11)

    print("Starting coarse grid search...")
    coarse = grid_search(df, corpus_tokens, qrels, k1_coarse, b_coarse, topk=args.topk)
    coarse.to_csv("bm25_grid_coarse.csv", index=False)

    best = coarse.sort_values("MAP@k", ascending=False).iloc[0]
    best_k1, best_b = best["k1"], best["b"]
    print("Best coarse:", best_k1, best_b)

    # refinement
    k1_fine = np.round(np.arange(max(0.1, best_k1-0.2), best_k1+0.201, 0.05), 3)
    b_fine  = np.round(np.arange(max(0.0, best_b-0.1), min(1.0, best_b+0.101), 0.02), 3)

    print("Starting fine grid search...")
    fine = grid_search(df, corpus_tokens, qrels, k1_fine, b_fine, topk=args.topk)
    fine.to_csv("bm25_grid_fine.csv", index=False)

    best_final = pd.concat([coarse, fine]).sort_values("MAP@k", ascending=False).iloc[0]
    print("Best final params:", best_final.to_dict())

    # heatmap (MAP)
    pivot = fine.pivot(index="k1", columns="b", values="MAP@k")
    plt.figure(figsize=(8,6))
    plt.title("MAP heatmap (fine grid)")
    plt.imshow(pivot.values, origin="lower", aspect="auto", cmap="viridis")
    plt.xticks(range(len(pivot.columns)), [f"{c:.2f}" for c in pivot.columns], rotation=90)
    plt.yticks(range(len(pivot.index)), [f"{i:.2f}" for i in pivot.index])
    plt.colorbar(label="MAP@k")
    plt.tight_layout()
    plt.savefig("bm25_map_heatmap.png", dpi=150)
    print("Saved bm25_map_heatmap.png")

usage: ipykernel_launcher.py [-h] [--corpus CORPUS] [--qrels QRELS]
                             [--topk TOPK]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Annisa\AppData\Roaming\jupyter\runtime\kernel-v389064b671650def22d887390682ec6c8f925f4be.json


FileNotFoundError: Qrels not found: ../output/qrels.csv — buat file qrels.csv atau berikan --qrels